# 数据清洗 — Taobao UserBehavior

**目标**:基于 `01_data_overview.ipynb` 发现的时间戳异常,把 5.5 万行脏数据过滤掉,
把结果固化为 **Parquet** 文件供后续切口 A / B / D 反复使用。

## 为什么写 Parquet 不留 CSV

| 维度 | CSV | Parquet |
|---|---|---|
| 文件大小 | 3.5 GB | ~1 GB(列式 + 压缩) |
| 查询速度 | 慢(逐行解析文本) | 快(列式 + 类型已知) |
| 类型保留 | 全是文本,每次都要重新解析 | 自带 schema |
| 兼容性 | 万能但低效 | DuckDB / Pandas / Spark 通用 |

后续切口 A/B/D 都会反复读这个文件,Parquet 能让每次查询快 5-10 倍。

## 输入与输出(数仓分层路径)

- **输入(ODS 层)**:`data/raw/ODS/UserBehavior.csv`(3.5 GB,1.0015 亿行,含约 5.5 万脏行)
- **输出(DWD 层)**:`data/processed/DWD/user_behavior_clean.parquet`(预期约 1 GB,1.0010 亿行,无脏行)

## 清洗规则

只剔除 **timestamp 不在 9 天窗口内** 的行(`2017-11-25 00:00:00 +08` 到 `2017-12-04 00:00:00 +08`,左闭右开)。
其他字段(`user_id` / `item_id` / `category_id` / `behavior_type`)在 01 notebook 已验证零缺失,不需要额外处理。

## 0. 环境与路径

In [ ]:
import duckdb
from pathlib import Path
import time

INPUT_PATH  = Path('../data/raw/ODS/UserBehavior.csv').resolve()
OUTPUT_PATH = Path('../data/processed/DWD/user_behavior_clean.parquet').resolve()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f'输入: {INPUT_PATH}  ({INPUT_PATH.stat().st_size / 1024**3:.2f} GB)')
print(f'输出: {OUTPUT_PATH}')

con = duckdb.connect()
print(f'DuckDB 版本: {duckdb.__version__}')

## 1. 定义清洗规则

把窗口边界设成 Python 常量,后面 SQL 用 f-string 插入。
让窗口定义**只在一处出现**,避免之后散落各处导致不一致。

In [ ]:
# 窗口: 2017-11-25 00:00:00 ~ 2017-12-04 00:00:00 (+08, 左闭右开)
WINDOW_START = 1511539200  # 2017-11-25 00:00:00 +08
WINDOW_END   = 1512316800  # 2017-12-04 00:00:00 +08

print(f'窗口起(含):   {WINDOW_START}  → 2017-11-25 00:00:00 +08')
print(f'窗口止(不含): {WINDOW_END}  → 2017-12-04 00:00:00 +08')
print(f'窗口长度: {(WINDOW_END - WINDOW_START) / 86400} 天')

## 2. 执行清洗:从 CSV 直接写 Parquet

用 DuckDB 的 `COPY (查询) TO 文件` 语法,**把过滤后的结果直接落盘,不需要先 CREATE TABLE 中转**,节省内存。

查询过程会扫描 1 亿行 CSV,带 WHERE 过滤,写入 Parquet。预计耗时 **2-5 分钟**,看硬盘速度。

In [ ]:
t0 = time.time()

con.execute(f"""
COPY (
    SELECT user_id, item_id, category_id, behavior_type, timestamp
    FROM read_csv(
        '{INPUT_PATH.as_posix()}',
        header=False,
        columns={{
            'user_id': 'BIGINT',
            'item_id': 'BIGINT',
            'category_id': 'BIGINT',
            'behavior_type': 'VARCHAR',
            'timestamp': 'BIGINT'
        }}
    )
    WHERE timestamp >= {WINDOW_START}
      AND timestamp <  {WINDOW_END}
) TO '{OUTPUT_PATH.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')
""")

elapsed = time.time() - t0
out_size = OUTPUT_PATH.stat().st_size / 1024**3
in_size  = INPUT_PATH.stat().st_size / 1024**3

print(f'清洗完成,耗时 {elapsed:.1f} 秒')
print(f'输入: {in_size:.2f} GB → 输出: {out_size:.2f} GB')
print(f'压缩比: {in_size / out_size:.2f}x')

## 3. 验证输出

### 3.1 把 Parquet 注册为视图

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW user_behavior_clean AS
SELECT * FROM '{OUTPUT_PATH.as_posix()}'
""")

con.sql("DESCRIBE user_behavior_clean").show()

### 3.2 行数、用户数、时间范围

In [ ]:
con.sql("""
SELECT
    COUNT(*)                       AS total_rows,
    COUNT(DISTINCT user_id)        AS unique_users,
    MIN(timestamp)                 AS min_ts,
    MAX(timestamp)                 AS max_ts,
    to_timestamp(MIN(timestamp))   AS min_datetime,
    to_timestamp(MAX(timestamp))   AS max_datetime
FROM user_behavior_clean
""").show()

**期望**:
- `total_rows` = **100,095,231**(与 01 notebook 算的窗口内行数完全一致)
- `unique_users` = **987,991**(减少了 3 个"全脏"用户)
- `min_datetime` ≥ `2017-11-25 00:00:00+08`,`max_datetime` < `2017-12-04 00:00:00+08`

### 3.3 每日行数(应该正好 9 行)

In [ ]:
con.sql("""
SELECT
    strftime(to_timestamp(timestamp), '%Y-%m-%d') AS date,
    COUNT(*)                                       AS rows,
    COUNT(DISTINCT user_id)                        AS users
FROM user_behavior_clean
GROUP BY date
ORDER BY date
""").show()

**期望**:正好 **9 行**,日期连续 2017-11-25 ~ 2017-12-03,行数与 01 notebook B 表完全一致。

### 3.4 行为类型分布(对照 01 notebook)

In [ ]:
con.sql("""
SELECT
    behavior_type,
    COUNT(*)                                            AS rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS pct
FROM user_behavior_clean
GROUP BY behavior_type
ORDER BY rows DESC
""").show()

**期望**:占比与 01 notebook 几乎一致(`pv ~89.6%` / `cart ~5.5%` / `fav ~2.9%` / `buy ~2.0%`)。
脏行只占总量 0.056%,过滤后比例不会有明显变化。

---

## 你的观察记录

> _TODO: 在这里写完上面所有结果之后的发现 / 疑问。_
>
> 比如:
> - Parquet 实际压缩比是多少?符合预期吗?
> - 总耗时多久?后续分析每次重读 Parquet 时间应该会快得多
> - 清洗后还有没有让你意外的地方?
> - 后续分析需要的字段都齐了吗?有没有想加的衍生字段(比如 `event_date`)?
